# Week 10 — Data Quality: NULLs, Duplicates & Deduplication
**Phase 4: Capstone Prep · ~15 minutes · Tables: `OLIST_ORDERS`, `OLIST_GEOLOCATION`**

---

## What you're practicing this week

Real data is messy. Before any analysis can be trusted you need to understand and handle data quality issues. This week you'll audit the dataset for NULLs and impossible values, and tackle the geolocation table which has significant duplication that needs resolving.

## New concepts this week

### Counting NULLs
`COUNT(*)` counts all rows. `COUNT(column)` skips NULLs. The gap between them = NULLs:

```sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(order_delivered_customer_date) AS null_count
FROM OLIST_ORDERS;
```

### COALESCE — replace NULLs with a fallback
```sql
-- Use actual delivery date, fall back to estimated if NULL
SELECT
    order_id,
    COALESCE(order_delivered_customer_date, order_estimated_delivery_date) AS effective_delivery
FROM OLIST_ORDERS;
```

### NULLIF — turn a value into NULL
```sql
-- Avoid divide-by-zero by turning 0 into NULL
SELECT revenue / NULLIF(order_count, 0) AS avg_revenue
```

### Deduplication with ROW_NUMBER + QUALIFY
```sql
-- Keep one row per zip code — the first one encountered
SELECT *
FROM OLIST_GEOLOCATION
QUALIFY ROW_NUMBER() OVER (PARTITION BY geolocation_zip_code_prefix ORDER BY 1) = 1;
```

---

## Try the examples

In [ ]:
-- Example 1: count NULLs across key columns
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(order_approved_at)              AS approved_nulls,
    COUNT(*) - COUNT(order_delivered_carrier_date)   AS carrier_nulls,
    COUNT(*) - COUNT(order_delivered_customer_date)  AS delivered_nulls
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS;

In [ ]:
-- Example 2: how many duplicate zip codes are in geolocation?
SELECT
    COUNT(*)                                     AS total_rows,
    COUNT(DISTINCT geolocation_zip_code_prefix)  AS distinct_zips,
    COUNT(*) - COUNT(DISTINCT geolocation_zip_code_prefix) AS duplicate_rows
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_GEOLOCATION;

In [ ]:
-- Example 3: deduplicate — keep one row per zip code
SELECT *
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_GEOLOCATION
QUALIFY ROW_NUMBER() OVER (PARTITION BY geolocation_zip_code_prefix ORDER BY 1) = 1
LIMIT 10;

---

## Explore


In [ ]:
SELECT * FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_GEOLOCATION LIMIT 10;

---

## Task 1 — NULL audit across OLIST_ORDERS

For each column in `OLIST_ORDERS`, calculate the number and percentage of NULL values. Return `column_name`, `null_count`, and `null_pct` rounded to 1 decimal place.

> **What to think about:** Which columns have the most NULLs? Does that make sense given what those columns represent?

In [ ]:
-- Task 1: your query here


In [ ]:
-- Submit Task 1
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    10,
    1,
    $$
        -- paste your Task 1 query here
    $$
);

---

## Task 2 — Find orders with impossible timestamps

Find orders where `order_delivered_customer_date` is earlier than `order_purchase_timestamp` — delivered before it was placed. Return `order_id`, `order_purchase_timestamp`, and `order_delivered_customer_date`.

> **What to think about:** These rows aren't NULLs — they have values, but the values are logically impossible. How would you handle them in a production pipeline?

In [ ]:
-- Task 2: your query here


In [ ]:
-- Submit Task 2
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    10,
    2,
    $$
        -- paste your Task 2 query here
    $$
);

---

## Task 3 — Deduplicate the geolocation table

Create a deduplicated version of `OLIST_GEOLOCATION` that keeps one row per `geolocation_zip_code_prefix`. Use `ROW_NUMBER()` with `QUALIFY`. Return all columns.

> **What to think about:** There's no single right answer for which row to keep. What criteria might you use to pick the most reliable one?

In [ ]:
-- Task 3: your query here


In [ ]:
-- Submit Task 3
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    10,
    3,
    $$
        -- paste your Task 3 query here
    $$
);

---

## Task 4 — Replace NULLs with sensible defaults

Use `COALESCE` to replace NULL values in `order_delivered_customer_date` with `order_estimated_delivery_date`. Return `order_id`, `order_status`, and a new column `effective_delivery_date`.

> **What to think about:** Is this substitution always valid? In what analysis contexts would it be misleading to treat the estimated date as the actual delivery date?

In [ ]:
-- Task 4: your query here


In [ ]:
-- Submit Task 4
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    10,
    4,
    $$
        -- paste your Task 4 query here
    $$
);

---

## Bonus challenge

Write a single data quality summary query that checks `OLIST_ORDERS` for: total rows, rows with any NULL, rows with impossible delivery timestamps, and duplicate `order_id`s. Return each metric as a named row.

In [ ]:
-- Bonus: your query here


---

*Next week: Query performance — understanding and optimising how Snowflake runs your queries.*